# Quantile Estimation

The median and the quartiles of a stream with `P2Quantile` from [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), the P² algorithm of Jain and Chlamtac (1985), which keeps five markers per quantile and stores no observations.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, Plots

Random.seed!(1)  # simulated: a stream of 10,000 draws from Exponential(1), whose mean and variance are 1 and median log(2) = 0.693
data = randexp(10_000)  # Replace data with your own stream of observations.

## The Median

One `P2Quantile` estimates one quantile. The estimate is close to the sample median of the stream, but not equal to it.

In [ ]:
o = fit!(P2Quantile(0.5), data)
println("Online median: ", value(o))
println("Batch median:  ", quantile(data, 0.5))

## The Three Quartiles in One Pass

A `Series` of three estimators reads the stream once.

In [ ]:
qs = fit!(Series(P2Quantile(0.25), P2Quantile(0.5), P2Quantile(0.75)), data)
println("Online quartiles: ", value(qs))
println("Batch quartiles:  ", quantile(data, [0.25, 0.5, 0.75]))

## How the Estimate Settles

The first five observations are stored as they are; after that the markers move one step at a time. The dashed line is the true median of Exponential(1), log(2).

In [ ]:
o = P2Quantile(0.5)
medians = map(x -> (fit!(o, x); value(o)), data)
plot(medians, xlabel="Observations", ylabel="Online median", label="P2Quantile")
hline!([log(2)], linestyle=:dash, label="log(2)")

## A Quantile in the Tail

Quantiles far in the tail need more observations before their markers settle. The 0.99 quantile of Exponential(1) is -log(0.01) = 4.605.

In [ ]:
for n in (100, 1_000, 10_000)
    tail = fit!(P2Quantile(0.99), data[1:n])
    println(n, " observations: ", value(tail))
end

## A Rolling Quantile

`P2Quantile` takes no weight. To follow a quantile that drifts, compute `quantile` on the values of a `MovingWindow`, at the cost of storing the window.

In [ ]:
window = fit!(MovingWindow(500, Float64), data)  # the last 500 observations
println("Median of the last 500 observations: ", quantile(value(window), 0.5))